# **Trabalho 2.1 da matéria de Matemática Computacional**

#### Observação: embora alguns valores de saída sejam diferentes dos valores dados nos slides, os métodos estão certos e estão encontrando a raiz corretamente.

---

### **Importações utilizadas nas respostas**:

In [1]:
%pip install pandas

import math
import pandas as pd
from IPython.display import display, Markdown
import time
import random

Note: you may need to restart the kernel to use updated packages.


---

## **Questão A**

### **Funções do DataFrame usadas nos exemplos 18 - 21**:

In [2]:
def format_scientific(value, precision):
    if value == 0: return "0.0000 x 10^0"
    
    formatted = "{:.{}e}".format(value, precision)
    
    return formatted.replace("e", " x 10^").replace("+", "")

def get_numerical_results_table(numerical_data):
    return pd.DataFrame({
        "Bisseção": numerical_data["bisection"],
        "Falsa Posição": numerical_data["false_position"],
        "Ponto Fixo": numerical_data["fixed_point"],
        "Newton": numerical_data["newton"],
        "Secante": numerical_data["secant"]
    }, index=[
        "Dados Iniciais",
        "x̄",
        "f(x̄)",
        "Erro em x",
        "Número de Iterações"
    ])

def get_computational_effort_table(effort_data):
    return pd.DataFrame({
        "Bisseção": effort_data["bisection"],
        "Falsa Posição": effort_data["false_position"],
        "Ponto Fixo": effort_data["fixed_point"],
        "Newton": effort_data["newton"],
        "Secante": effort_data["secant"]
    }, index=[
        "Operações por Iteração",
        "Complexidade de Operação",
        "Decisões Lógicas Totais",
        "Avaliações de Função por Iteração",
        "Número de Iterações"
    ])

def get_execution_time_table(time_data):
    return pd.DataFrame({
        "Bisseção": time_data["bisection"],
        "Falsa Posição": time_data["false_position"],
        "Ponto Fixo": time_data["fixed_point"],
        "Newton": time_data["newton"],
        "Secante": time_data["secant"]
    }, index=[
        "Tempo por Iteração (ms)",
        "Tempo Total (ms)"
    ])

def print_tables(numerical_data, effort_data, time_data):
    display(Markdown("## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes"))
    display(get_numerical_results_table(numerical_data))

    display(Markdown("## Tabela 2 – Análise de Esforço Computacional"))
    display(get_computational_effort_table(effort_data))

    display(Markdown("## Tabela 3 – Análise de Tempo de Execução"))
    display(get_execution_time_table(time_data))

---

### **Função do método de Bisseção**

#### O método da Bisseção funciona da seguinte maneira:

<center> 

![Bisseção](\imagens%20trabalho%202_1\bisection.png) 

</center>

Ele escolhe 2 pontos no eixo X (A e B), calcula a média entre os 2 pontos, armazena numa variável x e então calcula f(x).

Após isso ele verifica o sinal de f(x), se for negativo, então f(x) está mais perto de f(A), logo, o ponto A é substituído pelo valor da variável x.

Se o valor de f(x) for positivo, então f(x) está mais perto de f(B), logo, o ponto B é substituído pelo valor da variável x.

Esse processo é repetido ate a diferença entre A e B ou o valor de f(x) ser muito pequeno.

In [3]:
def bisection(function, interval, stopping_crit, precision, max_iterations = 100):
    #(1) initial values
    a = interval[0]
    b = interval[1]
    epsilon = stopping_crit

    #other initial values
    x = 0
    
    iterations, total_time = 0, 0
    dyn_ops, dyn_logic, dyn_evals = 0, 0, 0

    #(2) first verification
    dyn_ops += 1
    dyn_logic += 1
    if (b - a) < epsilon:
        x = random.uniform(a, b)

    else:
        #(3)
        iterations = 1
        
        #loop
        start_time = time.perf_counter()
        for i in range(max_iterations):
            #(4)
            dyn_evals += 1
            fa = function(a)

            #(5)
            dyn_ops += 2
            x = (a + b)/2

            dyn_evals += 1
            fx = function(x)
            
            #(6)
            dyn_ops += 1
            dyn_logic += 1
            if fa * fx > 0:
                a = x
            
            #(7)
            else:
                b = x

            #(8)
            dyn_ops += 1
            dyn_logic += 1
            if abs(b - a) < epsilon:
                #not choosing a random number in the range [a, b] for accuracy
                break
            
            #(9)
            iterations += 1
        
        #calculling the time
        final_time = time.perf_counter()
        total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        interval,                                   #initial data
        x,                                          #x̄
        format_scientific(function(x), precision),  #f(x̄)
        format_scientific(abs(b - a), precision),   #error
        iterations                                  #iterations
    ]

    #effort data
    effort_data = [
        dyn_ops // iterations if iterations > 0 else dyn_ops,     #operations per iteration
        "O(1)",                                                   #complexity
        dyn_logic,                                                #logical decisions
        dyn_evals // iterations if iterations > 0 else dyn_evals, #function evals per iteration
        iterations                                                #iterations
    ]

    #time data
    time_data = [
        total_time/iterations if iterations > 0 else 0, #time per iteration
        total_time                                      #total time
    ]

    return numerical_data, effort_data, time_data

---

### **Função do método de Posição Falsa**

#### O método da Posição Falsa funciona da seguinte maneira:

<center> 

![Bisseção](\imagens%20trabalho%202_1\false_position.png) 

</center>

Ele escolhe 2 pontos no eixo X (A e B), calcula f(A) e f(B) e traça uma reta entre esses 2 pontos.

Depois, ele verifica qual o valor x dessa reta que passa pelo y = 0, calculando f(x) logo em seguida.

Após isso, assim como no método de Bisseção, ele verifica o sinal de f(x) e substitui A ou B dependendo do sinal de f(x).

Esse processo é repetido ate a diferença entre A e B ou o valor de f(x) ser muito pequeno.

In [4]:
def false_position(function, interval, stopping_crit, precision, max_iterations = 100):
    #(1) initial values
    a = interval[0]
    b = interval[1]
    episolon_1 = stopping_crit
    episolon_2 = stopping_crit
    
    #other initial values
    x = 0
    
    iterations, total_time = 0, 0
    dyn_ops, dyn_logic, dyn_evals = 0, 0, 0

    #(2) first verification
    dyn_ops += 1
    dyn_logic += 1
    if (b - a) < episolon_1:
        x = random.uniform(a, b)
    
    else:
        dyn_evals += 1
        dyn_logic += 1
        if abs(function(a)) < episolon_2:
            x = a
        
        else:
            dyn_evals += 1
            dyn_logic += 1
            if abs(function(b)) < episolon_2:
                x = b

            else:
                #(3)
                iterations = 1

                #loop
                start_time = time.perf_counter()
                for i in range(max_iterations):
                    #(4)
                    dyn_evals += 1
                    fa = function(a)
                    dyn_evals += 1
                    fb = function(b)

                    #(5)
                    dyn_ops += 5
                    x = ((a * fb) - (b * fa))/(fb - fa)
                    
                    dyn_evals += 1
                    fx = function(x)
                    
                    #(6)
                    dyn_logic += 1
                    if abs(fx) < episolon_2:
                        break
                    
                    #(7)
                    dyn_ops += 1
                    dyn_logic += 1
                    if fa * fx > 0:
                        a = x

                    #(8)
                    else:
                        b = x

                    #(9)
                    dyn_ops += 1
                    dyn_logic += 1
                    if abs(b - a) < episolon_1:
                        x = random.uniform(a, b)
                        break

                    #(10)
                    iterations += 1
                
                #calculling the time
                final_time = time.perf_counter()
                total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        interval,                                     #initial data
        x,                                            #x̄
        format_scientific(function(x), precision),    #f(x̄)
        format_scientific(abs(b - a), precision),     #error
        iterations                                    #iterations
    ]

    #effort data
    effort_data = [
        dyn_ops // iterations if iterations > 0 else dyn_ops,     #operations per iteration
        "O(1)",                                                   #complexity
        dyn_logic,                                                #logical decisions
        dyn_evals // iterations if iterations > 0 else dyn_evals, #function evals per iteration
        iterations                                                #iterations
    ]

    #time data
    time_data = [
        total_time/iterations if iterations > 0 else 0, #time per iteration
        total_time                                      #total time
    ]

    return numerical_data, effort_data, time_data

---

### **Função do método do Ponto Fixo**

#### O método do Ponto Fixo funciona da seguinte maneira:

<center> 

![Bisseção](\imagens%20trabalho%202_1\fixed_point.png) 

</center>

Ele primeiro gera uma Função de Iteração $\phi(x)$ da função original, junto de uma função auxiliar y = x.

Onde essas duas funções de encontram, é onde está a raiz da função original.

Assim, ele dá um chute X inicial e calcula $\phi(X)$, após isso, ele pega um novo X1 = $\phi(X)$.

Esse processo é repetido ate a diferença entre Xn e Xn-1 ou o valor de f(Xn) ser muito pequeno.

In [5]:
def fixed_point(function, iteration_function, init_x, stopping_crit, precision, max_iterations = 100):
    #(1) initial values
    initial_x = init_x
    epsilon_1 = stopping_crit
    epsilon_2 = stopping_crit

    #other initial values
    x = initial_x
    x_1 = 0
    
    iterations, total_time, current_error = 0, 0, 0
    dyn_ops, dyn_logic, dyn_evals = 0, 0, 0

    #(2) first verification
    dyn_evals += 1
    dyn_logic += 1
    if abs(function(initial_x)) < epsilon_1:
        pass

    else:
        #(3)
        iterations = 1

        #loop
        start_time = time.perf_counter()
        for i in range(max_iterations):
            #(4)
            dyn_evals += 1
            x_1 = iteration_function(x)
            
            dyn_evals += 1
            fx_1 = function(x_1)

            #(5)
            dyn_ops += 1
            current_error = x_1 - x
            
            dyn_logic += 2
            if abs(fx_1) < epsilon_1 or abs(current_error) < epsilon_2:
                x = x_1
                break
            
            #(6)
            x = x_1

            #(7)
            iterations += 1

        #calculling the time
        final_time = time.perf_counter()
        total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        f"X0 = {initial_x}",                              #initial data
        x,                                                #x̄
        format_scientific(function(x), precision),        #f(x̄)
        format_scientific(abs(current_error), precision), #error
        iterations                                        #iterations
    ]

    #effort data
    effort_data = [
        dyn_ops // iterations if iterations > 0 else dyn_ops,     #operations per iteration
        "O(1)",                                                   #complexity
        dyn_logic,                                                #logical decisions
        dyn_evals // iterations if iterations > 0 else dyn_evals, #function evals per iteration
        iterations                                                #iterations
    ]

    #time data
    time_data = [
        total_time/iterations if iterations > 0 else 0, #time per iteration
        total_time                                      #total time
    ]

    return numerical_data, effort_data, time_data

---

### **Função do método de Newton-Raphson**

#### O método de Newton-Raphson funciona da seguinte maneira:

<center> 

![Bisseção](\imagens%20trabalho%202_1\newton.png) 

</center>

Ele dá um chute X inicial e calcula a tangente da função em f(X).

Após isso ele verifica qual o X1 que faz a reta tangente ser y = 0 e então usa esse X1 como novo X.

Esse processo é repetido ate a diferença entre Xn e Xn-1 ou o valor de f(Xn) ser muito pequeno.

In [6]:
def get_derivative_function(function):
    def derivative(x, h=1e-8):
        return (function(x + h) - function(x - h)) / (2 * h)
    
    return derivative

def newton(function, init_x, stopping_crit, precision, max_iterations = 100):
    derivative_function = get_derivative_function(function)

    #(1) initial values
    initial_x = init_x
    epsilon_1 = stopping_crit
    epsilon_2 = stopping_crit

    #other initial values
    x = initial_x
    x_1 = 0
    
    iterations, total_time, current_error = 0, 0, 0
    dyn_ops, dyn_logic, dyn_evals = 0, 0, 0

    #(2) first verification
    dyn_evals += 1
    dyn_logic += 1
    if abs(function(initial_x)) < epsilon_1:
        pass

    else:
        #(3)
        iterations = 1

        #loop
        start_time = time.perf_counter()
        for i in range(max_iterations):
            #(4)
            dyn_evals += 3
            dyn_ops += 2
            x_1 = x - (function(x)/derivative_function(x))
            
            dyn_evals += 1
            fx_1 = function(x_1)

            #(5)
            dyn_ops += 1
            current_error = x_1 - x
            
            dyn_logic += 2
            if abs(fx_1) < epsilon_1 or abs(current_error) < epsilon_2:
                x = x_1
                break
            
            #(6)
            x = x_1

            #(7)
            iterations += 1

        #calculling the time
        final_time = time.perf_counter()
        total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        f"X0 = {initial_x}",                                #initial data
        x,                                                  #x̄
        format_scientific(function(x), precision),          #f(x̄)
        format_scientific(abs(current_error), precision),   #error
        iterations                                          #iterations
    ]

    #effort data
    effort_data = [
        dyn_ops // iterations if iterations > 0 else dyn_ops,      #operations per iteration
        "O(1)",                                                    #complexity
        dyn_logic,                                                 #logical decisions
        dyn_evals // iterations if iterations > 0 else dyn_evals,  #function evals per iteration
        iterations                                                 #iterations
    ]

    #time data
    time_data = [
        total_time/iterations if iterations > 0 else 0, #time per iteration
        total_time                                      #total time
    ]

    return numerical_data, effort_data, time_data

---

### **Função do método da Secante**

#### O método da Secante funciona da seguinte maneira:

<center> 

![Bisseção](\imagens%20trabalho%202_1\secant.png) 

</center>

Ele dá um chute X0 e X1 inicial e calcula a secante entre f(X0) e f(X1).

Após isso ele verifica qual o X2 que faz a reta secante ser y = 0 e então substitui X0 = X1 e X1 = X2.

Esse processo é repetido ate a diferença entre Xn e Xn-1 ou o valor de f(Xn) ser muito pequeno.

In [7]:
def secant(function, ini_x_0, ini_x_1, stopping_crit, precision, max_iterations = 100):
    #(1) initial values
    initial_x_0 = ini_x_0
    initial_x_1 = ini_x_1
    epsilon_1 = stopping_crit
    epsilon_2 = stopping_crit

    #other initial values
    x = initial_x_1
    x_0 = initial_x_0
    x_1 = initial_x_1
    
    iterations, total_time, current_error = 0, 0, 0
    dyn_ops, dyn_logic, dyn_evals = 0, 0, 0

    #(2) first verification
    dyn_evals += 1
    dyn_logic += 1
    if abs(function(x_0)) < epsilon_1:
        x = x_0
    
    #(3) second verification
    else:
        dyn_evals += 1
        dyn_ops += 1
        dyn_logic += 2
        if abs(function(x_1)) < epsilon_1 or abs(x_1 - x_0) < epsilon_2:
            x = x_1
            
        else:
            #(4)
            iterations = 1

            #loop
            start_time = time.perf_counter()
            for i in range(max_iterations):
                dyn_evals += 1
                fx_0 = function(x_0)
                dyn_evals += 1
                fx_1 = function(x_1)

                #(5)
                dyn_ops += 5
                x_2 = x_1 - ((fx_1/(fx_1 - fx_0)) * (x_1 - x_0))
                
                dyn_evals += 1
                fx_2 = function(x_2)

                #(6)
                dyn_ops += 1
                current_error = x_2 - x_1
                
                dyn_logic += 2
                if abs(fx_2) < epsilon_1 or abs(current_error) < epsilon_2:
                    x = x_2
                    break
                
                #(7)
                x_0 = x_1
                x_1 = x_2

                #(7)
                iterations += 1

            #calculling the time
            final_time = time.perf_counter()
            total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        f"X0 = {initial_x_0}; X1 = {initial_x_1}",        #initial data
        x,                                                #x̄
        format_scientific(function(x), precision),        #f(x̄)
        format_scientific(abs(current_error), precision), #error
        iterations                                        #iterations
    ]
    
    #effort data
    effort_data = [
        dyn_ops // iterations if iterations > 0 else dyn_ops,      #operations per iteration
        "O(1)",                                                    #complexity
        dyn_logic,                                                 #logical decisions
        dyn_evals // iterations if iterations > 0 else dyn_evals,  #function evals per iteration
        iterations                                                 #iterations
    ]

    #time data
    time_data = [
        total_time/iterations if iterations > 0 else 0, #time per iteration
        total_time                                      #total time
    ]

    return numerical_data, effort_data, time_data

---

## Exemplos Questão A

#### Banco de dados inicialmente vazio:

In [8]:
numerical_data = {
    "bisection": [None, None, None, None, None],
    "false_position": [None, None, None, None, None],
    "fixed_point": [None, None, None, None, None],
    "newton": [None, None, None, None, None],
    "secant": [None, None, None, None, None]
}

effort_data = {
    "bisection": [None, None, None, None, None],
    "false_position": [None, None, None, None, None],
    "fixed_point": [None, None, None, None, None],
    "newton": [None, None, None, None, None],
    "secant": [None, None, None, None, None]
}

time_data = {
    "bisection": [None, None],
    "false_position": [None, None],
    "fixed_point": [None, None],
    "newton": [None, None],
    "secant": [None, None]
}

---

### **Exemplo 18**:

**Função Original:** 
$f(x) = e^{-x^2} - \cos(x)$

**Intervalo da Raiz:** 
$\xi \in (1, 2)$

**Precisão:** 
$\epsilon_1 = \epsilon_2 = 10^{-4}$

**Função de Iteração (Ponto Fixo):** 
$\phi(x) = \cos(x) - e^{-x^2} + x$

In [9]:
example_function = lambda x: (math.e**(-x**2)) - math.cos(x)
fixed_point_iteration_function = lambda x: math.cos(x) - math.e**(-x**2) + x
interval = [1, 2]
stopping_crit = 10**-4

numerical_data["bisection"], effort_data["bisection"], time_data["bisection"] = bisection(example_function, interval, stopping_crit, 4)
numerical_data["false_position"], effort_data["false_position"], time_data["false_position"] = false_position(example_function, interval, stopping_crit, 4)
numerical_data["fixed_point"], effort_data["fixed_point"], time_data["fixed_point"] = fixed_point(example_function, fixed_point_iteration_function, 1.5, stopping_crit, 4)
numerical_data["newton"], effort_data["newton"], time_data["newton"] = newton(example_function, 1.5, stopping_crit, 4)
numerical_data["secant"], effort_data["secant"], time_data["secant"] = secant(example_function, 1, 2, stopping_crit, 4)

print_tables(numerical_data, effort_data, time_data)

## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Dados Iniciais,"[1, 2]","[1, 2]",X0 = 1.5,X0 = 1.5,X0 = 1; X1 = 2
x̄,1.447449,1.447357,1.447525,1.447416,1.447413
f(x̄),2.1921 x 10^-05,-3.6388 x 10^-05,7.0258 x 10^-05,1.3204 x 10^-06,-5.2422 x 10^-07
Erro em x,6.1035 x 10^-05,5.5289 x 10^-01,1.9319 x 10^-04,1.7072 x 10^-03,1.8553 x 10^-04
Número de Iterações,14,6,6,2,5


## Tabela 2 – Análise de Esforço Computacional

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Operações por Iteração,4,6,1,3,6
Complexidade de Operação,O(1),O(1),O(1),O(1),O(1)
Decisões Lógicas Totais,29,19,13,5,13
Avaliações de Função por Iteração,2,3,2,4,3
Número de Iterações,14,6,6,2,5


## Tabela 3 – Análise de Tempo de Execução

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Tempo por Iteração (ms),0.002321,0.002333,0.001717,0.00355,0.00204
Tempo Total (ms),0.032500,0.014000,0.010300,0.00710,0.01020


#### **Análise do exemplo 18**

* **Tempo por Iteração:** O método do **Ponto Fixo** foi o mais rápido por iteração (**0.000967 ms**), enquanto o método de **Newton** foi o mais lento por iteração (**0.00215 ms**).
* **Tempo Total:** O método de **Newton** foi o mais rápido em tempo total (**0.00430 ms**), precisando de apenas 2 iterações para convergir.
* **Ineficiência:** A **Bisseção** apresentou o pior tempo total (**0.0210 ms**), sendo quase 4 vezes mais lenta que o Newton.
* **Número de Operações:** O **Ponto Fixo** é o método de menor complexidade intrínseca, realizando apenas **1 operação** por iteração.
* **Avaliações de Função:** O método de **Newton** possui o maior número de avaliações por iteração (**4**), ao contrário da Bisseção e do Ponto Fixo que possuem apenas **2** .
* **Decisões Lógicas:** A **Bisseção** é o método que mais sobrecarrega o fluxo lógico, com **29 decisões** totais, ao contrário do Newton, que precisou de apenas **5 decisões** no total.

---

### **Exemplo 19**

**Função Original:** $f(x) = x^3 - x - 1$

**Intervalo da Raiz:** $\xi \in (1, 2)$

**Precisão:** $\epsilon_1 = \epsilon_2 = 10^{-6}$

**Função de Iteração (Ponto Fixo):** $\phi(x) = (x + 1)^{1/3}$

In [10]:
example_function = lambda x: x**3 - x - 1
fixed_point_iteration_function = lambda x: (x + 1)**(1/3)
interval = [1, 2]
stopping_crit = 10**-6

numerical_data["bisection"], effort_data["bisection"], time_data["bisection"] = bisection(example_function, interval, stopping_crit, 7)
numerical_data["false_position"], effort_data["false_position"], time_data["false_position"] = false_position(example_function, interval, stopping_crit, 7)
numerical_data["fixed_point"], effort_data["fixed_point"], time_data["fixed_point"] = fixed_point(example_function, fixed_point_iteration_function, 1, stopping_crit, 7)
numerical_data["newton"], effort_data["newton"], time_data["newton"] = newton(example_function, 0, stopping_crit, 7)
numerical_data["secant"], effort_data["secant"], time_data["secant"] = secant(example_function, 0, .5, stopping_crit, 7)

print_tables(numerical_data, effort_data, time_data)

## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Dados Iniciais,"[1, 2]","[1, 2]",X0 = 1,X0 = 0,X0 = 0; X1 = 0.5
x̄,1.324718,1.324718,1.324718,1.324718,1.324718
f(x̄),-1.8575764 x 10^-06,-8.2906613 x 10^-07,-4.7372647 x 10^-07,2.7204905 x 10^-12,-4.3405755 x 10^-08
Erro em x,9.5367432 x 10^-07,6.7528250 x 10^-01,4.7372647 x 10^-07,8.2941372 x 10^-07,1.1916614 x 10^-05
Número de Iterações,20,17,9,21,26


## Tabela 2 – Análise de Esforço Computacional

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Operações por Iteração,4,6,1,3,6
Complexidade de Operação,O(1),O(1),O(1),O(1),O(1)
Decisões Lógicas Totais,41,52,19,43,55
Avaliações de Função por Iteração,2,3,2,4,3
Número de Iterações,20,17,9,21,26


## Tabela 3 – Análise de Tempo de Execução

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Tempo por Iteração (ms),0.0012,0.001341,0.001333,0.001557,0.001042
Tempo Total (ms),0.0240,0.022800,0.012000,0.032700,0.027100


#### **Análise do exemplo 19**

* **Tempo por Iteração:** O método da **Bisseção** foi o mais rápido por iteração (**0.00102 ms**), enquanto o método de **Newton** foi o mais lento por iteração (**0.001543 ms**).
* **Tempo Total:** O método do **Ponto Fixo** foi o mais rápido em tempo total (**0.0099 ms**), precisando de apenas 9 iterações para convergir.
* **Ineficiência:** O **Newton** apresentou o pior tempo total (**0.032400 ms**), sendo mais de 3 vezes mais lenta que o Ponto Fixo.
* **Número de Operações:** O **Ponto Fixo** é o método de menor complexidade intrínseca, realizando apenas **1 operação** por iteração.
* **Avaliações de Função:** O método de **Newton** possui o maior número de avaliações por iteração (**4**), ao contrário da Bisseção e do Ponto Fixo que possuem apenas **2**.
* **Decisões Lógicas:** A **Secante** é o método que mais sobrecarrega o fluxo lógico, com **55 decisões** totais, ao contrário do Ponto Fixo, que precisou de apenas **19 decisões** no total.

---

### **Exemplo 20**:

**Função Original:** $f(x) = 4\text{sen}(x) - e^x$

**Intervalo da Raiz:** $\xi \in (0, 1)$

**Precisão:** $\epsilon_1 = \epsilon_2 = 10^{-5}$

**Função de Iteração (Ponto Fixo):** $\phi(x) = x - 2\text{sen}(x) + 0.5e^x$

In [11]:
example_function = lambda x: 4 * math.sin(x) - math.e**x
fixed_point_iteration_function = lambda x: x - 2 * math.sin(x) + .5 * math.e**x
interval = [0, 1]
stopping_crit = 10**-5

numerical_data["bisection"], effort_data["bisection"], time_data["bisection"] = bisection(example_function, interval, stopping_crit, 4)
numerical_data["false_position"], effort_data["false_position"], time_data["false_position"] = false_position(example_function, interval, stopping_crit, 4)
numerical_data["fixed_point"], effort_data["fixed_point"], time_data["fixed_point"] = fixed_point(example_function, fixed_point_iteration_function, .5, stopping_crit, 4)
numerical_data["newton"], effort_data["newton"], time_data["newton"] = newton(example_function, .5, stopping_crit, 4)
numerical_data["secant"], effort_data["secant"], time_data["secant"] = secant(example_function, 0, 1, stopping_crit, 4)

print_tables(numerical_data, effort_data, time_data)

## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Dados Iniciais,"[0, 1]","[0, 1]",X0 = 0.5,X0 = 0.5,X0 = 0; X1 = 1
x̄,0.370552,0.370559,0.370556,0.370558,0.370558
f(x̄),-1.3755 x 10^-05,1.6698 x 10^-06,-4.5194 x 10^-06,-2.7836 x 10^-08,5.2605 x 10^-09
Erro em x,7.6294 x 10^-06,3.7056 x 10^-01,1.6144 x 10^-05,1.3863 x 10^-04,5.7406 x 10^-06
Número de Iterações,17,8,5,3,7


## Tabela 2 – Análise de Esforço Computacional

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Operações por Iteração,4,6,1,3,6
Complexidade de Operação,O(1),O(1),O(1),O(1),O(1)
Decisões Lógicas Totais,35,25,11,7,17
Avaliações de Função por Iteração,2,3,2,4,3
Número de Iterações,17,8,5,3,7


## Tabela 3 – Análise de Tempo de Execução

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Tempo por Iteração (ms),0.001335,0.001062,0.00094,0.001567,0.0009
Tempo Total (ms),0.022700,0.008500,0.00470,0.004700,0.0063


#### **Análise do exemplo 20**

* **Tempo por Iteração:** O método da **Secante** foi o mais rápido por iteração (**0.000957 ms**), enquanto o método da **Falsa Posição** foi o mais lento por iteração (**0.001663 ms**).
* **Tempo Total:** O método de **Newton** foi o mais rápido em tempo total (**0.004400 ms**), precisando de apenas 3 iterações para convergir.
* **Ineficiência:** A **Bisseção** apresentou o pior tempo total (**0.022200 ms**), sendo mais de 4 vezes mais lenta que o Ponto Fixo.
* **Número de Operações:** O **Ponto Fixo** é o método de menor complexidade intrínseca, realizando apenas **1 operação** por iteração.
* **Avaliações de Função:** O método de **Newton** possui o maior número de avaliações por iteração (**4**), ao contrário da Bisseção e do Ponto Fixo que possuem apenas **2**.
* **Decisões Lógicas:** A **Bisseção** é o método que mais sobrecarrega o fluxo lógico, com **35 decisões** totais, ao contrário do Newton, que precisou de apenas **7 decisões** no total.

---

### **Exemplo 21**:

**Função Original:** $f(x) = x \log(x) - 1$

**Intervalo da Raiz:** $\xi \in (2, 3)$

**Precisão:** $\epsilon_1 = \epsilon_2 = 10^{-7}$

**Função de Iteração (Ponto Fixo):** $\phi(x) = x - 1.3(x \log x - 1)$

In [12]:
example_function = lambda x: (x * math.log10(x)) - 1
fixed_point_iteration_function = lambda x: x - 1.3 * ((x * math.log10(x)) - 1)
interval = [2, 3]
stopping_crit = 10**-7

numerical_data["bisection"], effort_data["bisection"], time_data["bisection"] = bisection(example_function, interval, stopping_crit, 4)
numerical_data["false_position"], effort_data["false_position"], time_data["false_position"] = false_position(example_function, interval, stopping_crit, 4)
numerical_data["fixed_point"], effort_data["fixed_point"], time_data["fixed_point"] = fixed_point(example_function, fixed_point_iteration_function, 2.5, stopping_crit, 4)
numerical_data["newton"], effort_data["newton"], time_data["newton"] = newton(example_function, 2.5, stopping_crit, 4)
numerical_data["secant"], effort_data["secant"], time_data["secant"] = secant(example_function, 2.3, 2.7, stopping_crit, 4)

print_tables(numerical_data, effort_data, time_data)

## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Dados Iniciais,"[2, 3]","[2, 3]",X0 = 2.5,X0 = 2.5,X0 = 2.3; X1 = 2.7
x̄,2.506184,2.506184,2.506184,2.506184,2.506184
f(x̄),1.2600 x 10^-08,-9.9280 x 10^-08,2.0508 x 10^-08,1.3518 x 10^-12,2.9153 x 10^-08
Erro em x,5.9605 x 10^-08,4.9382 x 10^-01,3.2006 x 10^-07,3.9882 x 10^-06,8.0561 x 10^-05
Número de Iterações,24,5,5,2,3


## Tabela 2 – Análise de Esforço Computacional

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Operações por Iteração,4,6,1,3,6
Complexidade de Operação,O(1),O(1),O(1),O(1),O(1)
Decisões Lógicas Totais,49,16,11,5,9
Avaliações de Função por Iteração,2,3,2,4,3
Número de Iterações,24,5,5,2,3


## Tabela 3 – Análise de Tempo de Execução

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Tempo por Iteração (ms),0.000704,0.00092,0.00072,0.00135,0.0008
Tempo Total (ms),0.016900,0.00460,0.00360,0.00270,0.0024


#### **Análise do exemplo 21**

* **Tempo por Iteração:** O método do **Ponto Fixo** foi o mais rápido por iteração (**0.00122 ms**), enquanto o método de **Newton** foi o mais lento por iteração (**0.00245 ms**).
* **Tempo Total:** O método da **Secante** foi o mais rápido em tempo total (**0.004300 ms**), precisando de apenas 3 iterações para convergir.
* **Ineficiência:** A **Bisseção** apresentou o pior tempo total (**0.032500 ms**), sendo mais de 7 vezes mais lenta que a Secante.
* **Número de Operações:** O **Ponto Fixo** é o método de menor complexidade intrínseca, realizando apenas **1 operação** por iteração.
* **Avaliações de Função:** O método de **Newton** possui o maior número de avaliações por iteração (**4**), ao contrário da Bisseção e do Ponto Fixo que possuem apenas **2**.
* **Decisões Lógicas:** A **Bisseção** é o método que mais sobrecarrega o fluxo lógico, com **49 decisões** totais, ao contrário do Newton, que precisou de apenas **5 decisões** no total.

---

### **Exemplo 22**:

**Função Original:** $f(x) = x^3 - 3.5x^2 + 4x - 1.5 = (x - 1)^2 (x - 1.5)$.

**Raízes:** $\xi_1 = 1$ (raiz dupla) e $\xi_2 = 1.5$

**Precisão:** $\epsilon_1 = \epsilon_2 = 10^{-7}$ 

In [13]:
example_function = lambda x: x**3 - 3.5*x**2 + 4*x - 1.5
stopping_crit = 10**-7

def run_example_22_comparison():
    newton_num = {}
    newton_eff = {}
    newton_time = {}
    
    test_points = [0.5, 1.33333, 1.33334]
    test_names = ["Teste 1", "Teste 2", "Teste 3"]

    for x0, name in zip(test_points, test_names):
        numerical_results, effort_results, time_results = newton(example_function, x0, stopping_crit, 4)
        
        newton_num[name] = numerical_results
        newton_eff[name] = effort_results
        newton_time[name] = time_results

    print_example_22_tables(newton_num, newton_eff, newton_time)

def print_example_22_tables(num_data, eff_data, time_data):
    display(Markdown("## Tabela 1 – Resultados Numéricos (Comparação do Método de Newton)"))
    data_frame_num = pd.DataFrame(num_data, index=[
        "Dado Inicial (x0)", 
        "Raiz Aproximada (x̄)", 
        "f(x̄)", 
        "Erro em x", 
        "Número de Iterações"
    ])

    display(data_frame_num)

    display(Markdown("## Tabela 2 – Análise de Esforço Computacional"))
    data_frame_eff = pd.DataFrame(eff_data, index=[
        "Operações por Iteração", 
        "Complexidade de Operação", 
        "Decisões Lógicas Totais", 
        "Avaliações de Função por Iteração", 
        "Número Total de Iterações"
    ])

    display(data_frame_eff)

    display(Markdown("## Tabela 3 – Análise de Tempo de Execução"))
    data_frame_tim = pd.DataFrame(time_data, index=[
        "Tempo por Iteração (ms)",
        "Tempo Total (ms)"
    ])

    display(data_frame_tim)

run_example_22_comparison()

## Tabela 1 – Resultados Numéricos (Comparação do Método de Newton)

,Teste 1,Teste 2,Teste 3
Dado Inicial (x0),X0 = 0.5,X0 = 1.33333,X0 = 1.33334
Raiz Aproximada (x̄),0.999553,0.999702,1.5
f(x̄),-9.9934 x 10^-08,-4.4473 x 10^-08,1.5345 x 10^-09
Erro em x,4.4608 x 10^-04,2.9777 x 10^-04,3.9172 x 10^-05
Número de Iterações,11,35,27


## Tabela 2 – Análise de Esforço Computacional

,Teste 1,Teste 2,Teste 3
Operações por Iteração,3,3,3
Complexidade de Operação,O(1),O(1),O(1)
Decisões Lógicas Totais,23,71,55
Avaliações de Função por Iteração,4,4,4
Número Total de Iterações,11,35,27


## Tabela 3 – Análise de Tempo de Execução

,Teste 1,Teste 2,Teste 3
Tempo por Iteração (ms),0.001482,0.00116,0.001052
Tempo Total (ms),0.016300,0.04060,0.028400


#### **Análise do exemplo 22**

* **Tempo por Iteração:** O **Teste 3** foi o mais rápido por iteração (**0.001952 ms**), enquanto o **Teste 1** foi o mais lento por iteração (**0.002491 ms**).
* **Tempo Total:** O **Teste 1** foi o mais rápido em tempo total (**0.027400 ms**), precisando de apenas 11 iterações para convergir.
* **Ineficiência:** O **Teste 2** apresentou o pior tempo total (**0.071000 ms**), sendo mais de 2,5 vezes mais lento que o Teste 1.
* **Número de Operações:** Todos os testes mantiveram a mesma complexidade, realizando **3 operações** por iteração.
* **Avaliações de Função:** Todos os testes realizaram o mesmo número de avaliações por iteração (**4**).
* **Decisões Lógicas:** O **Teste 2** foi o que mais sobrecarregou o fluxo lógico, com **71 decisões** totais, ao contrário do Teste 1, que precisou de apenas **23 decisões** no total.

---

## **Questão B**

### **Funções do DataFrame usadas na questão B**:

In [14]:
def get_task_b_numerical_table(numerical_data):
    return pd.DataFrame({
        "Newton Polinomial": numerical_data["newton"],
        "Secante Adaptada": numerical_data["secant"],
        "Ponto Fixo Adaptada": numerical_data["fixed_point"]
    }, index=[
        "Dados Iniciais",
        "x̄",
        "f(x̄)",
        "Erro em x",
        "Número de Iterações"
    ])

def get_task_b_effort_table(effort_data):
    return pd.DataFrame({
        "Newton Polinomial": effort_data["newton"],
        "Secante Adaptada": effort_data["secant"],
        "Ponto Fixo Adaptada": effort_data["fixed_point"]
    }, index=[
        "Operações por Iteração",
        "Complexidade de Operação",
        "Decisões Lógicas Totais",
        "Avaliações de Função por Iteração",
        "Número de Iterações"
    ])

def get_task_b_time_table(time_data):
    return pd.DataFrame({
        "Newton Polinomial": time_data["newton"],
        "Secante Adaptada": time_data["secant"],
        "Ponto Fixo Adaptada": time_data["fixed_point"]
    }, index=[
        "Tempo por Iteração (ms)",
        "Tempo Total (ms)"
    ])

def print_task_b_tables(numerical_data, effort_data, time_data):
    display(Markdown("## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes"))
    display(get_task_b_numerical_table(numerical_data))

    display(Markdown("## Tabela 2 – Análise de Esforço Computacional"))
    display(get_task_b_effort_table(effort_data))

    display(Markdown("## Tabela 3 – Análise de Tempo de Execução"))
    display(get_task_b_time_table(time_data))

---

### **Newton-Horner com polinômios (slide 16 e 17)**
#### Este Método será utilizado na adaptação dos próximos 2 métodos.

In [15]:
def newton_horner(coefficients, init_x, stopping_crit, precision, max_iterations = 100):
    #initial values
    initial_x = init_x
    epsilon = stopping_crit

    x = initial_x
    n = len(coefficients) - 1

    iterations, total_time, current_error = 0, 0, 0
    dyn_ops, dyn_logic, dyn_evals = 0, 0, 0

    #(1)
    delta_x = x

    #(2) loop
    iterations = 1
    start_time = time.perf_counter()
    for k in range(max_iterations):
        b = coefficients[n]                 #b = an
        c = b                               #c = b

        for i in range(n - 1, 0, -1):
            dyn_ops += 2
            b = coefficients[i] + b * x     #b = ai + b*x
            
            dyn_ops += 2
            c = b + c * x                   #c = b + c*x

        dyn_ops += 2
        b = coefficients[0] + b * x         #b = a0 + b*x
        
        dyn_logic += 1
        if abs(b) < epsilon:                #first verification
            break
                
        dyn_ops += 1
        delta_x = b / c                     #deltax = b/c

        dyn_ops += 1
        x -= delta_x                        #x = x - deltax
        
        current_error = delta_x
        
        dyn_logic += 1
        if abs(delta_x) < epsilon:          #second verification
            break

        iterations += 1

    #calculling the time
    final_time = time.perf_counter()
    total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        f"X0 = {initial_x}",                              #initial data
        x,                                                #x̄
        format_scientific(b, precision),                  #f(x̄)
        format_scientific(abs(current_error), precision), #error
        iterations                                        #iterations
    ]

    #effort data
    effort_data = [
        dyn_ops // iterations if iterations > 0 else 0,             #operations per iteration
        f"O({n})",                                                  #complexity
        dyn_logic,                                                  #logical decisions
        dyn_evals // iterations if iterations > 0 else dyn_evals,   #function evals
        iterations                                                  #iterations
    ]

    #time data
    time_data = [
        total_time/iterations if iterations > 0 else 0,   #time per iteration
        total_time                                        #total time
    ]

    return numerical_data, effort_data, time_data

### **Método da Secante (Adaptado)**: 

* **Funcionamento**: A Secante utiliza a diferença entre dois pontos ($x_k$ e $x_{k-1}$) para aproximar a derivada.
* **A Adaptação**: Foi adicionada uma função auxiliar horner_eval. O valor de $f(x_{0})$ é armazenado em uma variável de memória (`fx_0`).
* **Lógica Interna**: A cada iteração, o polinômio é processado apenas uma vez para o novo ponto, evitando recálculo desnecessário do ponto anterior.

In [16]:
def horner_eval(n, coefficients, x_val):
    b = coefficients[n]
    for i in range(n - 1, -1, -1):
        b = coefficients[i] + b * x_val

    return b

def secant_horner(coefficients, init_x0, init_x1, stopping_crit, precision, max_iterations = 100):
    #initial values
    x_0 = init_x0
    x_1 = init_x1
    
    epsilon = stopping_crit
    n = len(coefficients) - 1

    current_error = abs(x_1 - x_0)
    iterations = 0
    dyn_ops, dyn_logic, dyn_evals = 0, 0, 0
    
    fx_0 = horner_eval(n, coefficients, x_0)
    fx_1 = 0

    #loop
    iterations = 1
    start_time = time.perf_counter()
    for k in range(1, max_iterations + 1):
        dyn_evals += 1
        dyn_ops += 2
        fx_1 = horner_eval(n, coefficients, x_1) 
        
        dyn_logic += 1
        if abs(fx_1) < epsilon:
            break
            
        dyn_ops += 1
        denominator = fx_1 - fx_0
            
        dyn_ops += 3
        delta_x = (fx_1 * (x_1 - x_0)) / denominator
        
        x_0 = x_1
        fx_0 = fx_1

        dyn_ops += 1
        x_1 -= delta_x
        
        current_error = abs(delta_x)
        
        dyn_logic += 1
        if current_error < epsilon:
            break

        iterations += 1

    #calculling the time
    final_time = time.perf_counter()
    total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        f"X0={init_x0}, X1={init_x1}",                    #initial data
        x_1,                                              #x̄
        format_scientific(fx_1, precision),               #f(x̄)
        format_scientific(abs(current_error), precision), #error
        iterations                                        #iterations
    ]

    #effort data
    effort_data = [
        dyn_ops // iterations if iterations > 0 else 0,             #operations per iteration
        f"O({n})",                                                  #complexity
        dyn_logic,                                                  #logical decisions
        dyn_evals // iterations if iterations > 0 else dyn_evals,   #function evals
        iterations                                                  #iterations
    ]

    #time data
    time_data = [
        total_time/iterations if iterations > 0 else 0,   #time per iteration
        total_time                                        #total time
    ]

    return numerical_data, effort_data, time_data

### **Método do Ponto Fixo (Adaptado)**:

* **Funcionamento**: Baseia-se na convergência de $x_{k+1} = \phi(x_k)$. O polinômio original não costuma ser avaliado durante o loop de iteração.
* **A Adaptação**: A adaptação ocorreu no encerramento da função. Após encontrar a raiz $\bar{x}$ via função de iteração, o Horner é chamado uma única vez.
* **Lógica Interna**: Esta chamada final serve para calcular $f(\bar{x})$, para comparar o erro deste método com os de Newton e Secante na mesma escala.

In [17]:
def horner_eval(n, coefficients, x_val):
    b = coefficients[n]
    for i in range(n - 1, -1, -1):
        b = coefficients[i] + b * x_val

    return b

def fixed_point_horner(coefficients, iteration_function, init_x, stopping_crit, precision, max_iterations=100):
    #initial values
    initial_x = init_x
    epsilon = stopping_crit
    x = initial_x
    n = len(coefficients) - 1
    
    iterations = 0
    dyn_ops, dyn_logic, dyn_evals = 0, 0, 0

    #loop
    iterations = 1
    start_time = time.perf_counter()
    for k in range(1, max_iterations + 1):
        dyn_evals += 1
        x_new = iteration_function(x)
        
        dyn_ops += 1
        current_error = abs(x_new - x)
        
        x = x_new

        dyn_logic += 1
        if current_error < epsilon:
            break

        iterations += 1

    b_final = horner_eval(n, coefficients, x)

    #calculling the time
    final_time = time.perf_counter()
    total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        f"X0 = {initial_x}",                              #initial data
        x,                                                #x̄
        format_scientific(b_final, precision),            #f(x̄)
        format_scientific(abs(current_error), precision), #error
        iterations                                        #iterations
    ]

    #effort data
    effort_data = [
        dyn_ops // iterations if iterations > 0 else 0,             #operations per iteration
        "O(1)",                                                     #complexity
        dyn_logic,                                                  #logical decisions
        dyn_evals // iterations if iterations > 0 else dyn_evals,   #function evals
        iterations                                                  #iterations
    ]

    #time data
    time_data = [
        total_time/iterations if iterations > 0 else 0,   #time per iteration
        total_time                                        #total time
    ]

    return numerical_data, effort_data, time_data

---

### **Funções do Exemplo 1**:

**Função Polinomial**: $p_5(x) = x^5 - 3.7x^4 + 7.4x^3 - 10.8x^2 + 10.8x - 6.8 = 0$.

**Função de Iteração**: $\phi(x) = \sqrt[5]{3.7x^4 - 7.4x^3 + 10.8x^2 - 10.8x + 6.8}$.

---

### **Exemplo 1**

In [18]:
coefficients_example_1 = [1, -3.7, 7.4, -10.8, 10.8, -6.8][::-1]
fixed_point_iteration_function_example_1 = lambda x: (3.7*x**4 - 7.4*x**3 + 10.8*x**2 - 10.8*x + 6.8)**(1/5)
stopping_crit = 10**-6

newton_numerical_results, newton_effort_results, newton_time_results = newton_horner(coefficients_example_1, 1.5, stopping_crit, 4)

secant_numerical_results, secant_effort_results, secant_time_results = secant_horner(coefficients_example_1, 1.5, 1.6, stopping_crit, 4)

fixed_point_numerical_results, fixed_point_effort_results, fixed_point_time_results = fixed_point_horner(coefficients_example_1, fixed_point_iteration_function_example_1, 1.5, stopping_crit, 4)

numerical_data = {
    "newton": newton_numerical_results, 
    "secant": secant_numerical_results, 
    "fixed_point": fixed_point_numerical_results
}

effort_data = {
    "newton": newton_effort_results, 
    "secant": secant_effort_results, 
    "fixed_point": fixed_point_effort_results
}

time_data = {
    "newton": newton_time_results, 
    "secant": secant_time_results, 
    "fixed_point": fixed_point_time_results
}

print_task_b_tables(numerical_data, effort_data, time_data)

## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Dados Iniciais,X0 = 1.5,"X0=1.5, X1=1.6",X0 = 1.5
x̄,1.7,1.7,1.699995
f(x̄),4.1855 x 10^-07,-2.5515 x 10^-08,-3.3358 x 10^-05
Erro em x,1.8742 x 10^-04,4.9002 x 10^-06,9.6761 x 10^-07
Número de Iterações,5,6,55


## Tabela 2 – Análise de Esforço Computacional

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Operações por Iteração,19,6,1
Complexidade de Operação,O(5),O(5),O(1)
Decisões Lógicas Totais,9,11,55
Avaliações de Função por Iteração,0,1,1
Número de Iterações,5,6,55


## Tabela 3 – Análise de Tempo de Execução

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Tempo por Iteração (ms),0.002,0.00105,0.00048
Tempo Total (ms),0.010,0.00630,0.02640


#### **Análise Exemplo 1**

* **Tempo por Iteração:** O método do **Ponto Fixo Adaptado** foi o mais rápido por iteração, enquanto o método **Newton Polinomial** foi o mais lento por iteração.
* **Tempo Total:** O método da **Secante Adaptada** foi o mais rápido em tempo total, precisando de apenas 6 iterações para convergir.
* **Ineficiência:** O **Ponto Fixo Adaptado** apresentou o pior tempo total, sendo algumas vezes mais lento que o método da Secante Adaptada.
* **Número de Operações:** O **Ponto Fixo Adaptado** é o método de menor complexidade intrínseca, realizando apenas **1 operação** por iteração.
* **Avaliações de Função:** O método do **Ponto Fixo Adaptado** e da **Secante Adaptada** possuem os maiores números de avaliações totais (**1**).
* **Decisões Lógicas:** O **Ponto Fixo Adaptado** é o método que mais sobrecarrega o fluxo lógico, com **55 decisões** totais, ao contrário do Newton, que precisou de apenas **9 decisões** no total.

---

### **Funções do Exemplo 2**:

**Função Polinomial**: $p_3(x) = x^3 - 3x + 3 = 0$.

**Função de Iteração**: $\phi(x) = \sqrt[3]{3x - 3}$.

---

### **Exemplo 2.1**

In [19]:
coefficients_example_2 = [1, 0, -3, 3][::-1]
fixed_point_iteration_function_example_2 = lambda x: math.cbrt(3*x - 3)
stopping_crit = 10**-6

newton_numerical_results, newton_effort_results, newton_time_results = newton_horner(coefficients_example_2, -0.8, stopping_crit, 4, max_iterations=30)

secant_numerical_results, secant_effort_results, secant_time_results = secant_horner(coefficients_example_2, -0.8, -0.9, stopping_crit, 4, max_iterations=30)

fixed_point_numerical_results, fixed_point_effort_results, fixed_point_time_results = fixed_point_horner(coefficients_example_2, fixed_point_iteration_function_example_2, -0.8, stopping_crit, 4, max_iterations=30)

numerical_data = {
    "newton": newton_numerical_results, 
    "secant": secant_numerical_results, 
    "fixed_point": fixed_point_numerical_results
}

effort_data = {
    "newton": newton_effort_results, 
    "secant": secant_effort_results, 
    "fixed_point": fixed_point_effort_results
}

time_data = {
    "newton": newton_time_results, 
    "secant": secant_time_results, 
    "fixed_point": fixed_point_time_results
}

print_task_b_tables(numerical_data, effort_data, time_data)

## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Dados Iniciais,X0 = -0.8,"X0=-0.8, X1=-0.9",X0 = -0.8
x̄,-2.103803,-2.103803,-2.103803
f(x̄),-7.6802 x 10^-10,2.9609 x 10^-08,1.3097 x 10^-06
Erro em x,1.1031 x 10^-05,6.3818 x 10^-06,4.3657 x 10^-07
Número de Iterações,18,12,11


## Tabela 2 – Análise de Esforço Computacional

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Operações por Iteração,11,6,1
Complexidade de Operação,O(3),O(3),O(1)
Decisões Lógicas Totais,35,23,11
Avaliações de Função por Iteração,0,1,1
Número de Iterações,18,12,11


## Tabela 3 – Análise de Tempo de Execução

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Tempo por Iteração (ms),0.000811,0.001075,0.002309
Tempo Total (ms),0.014600,0.012900,0.025400


#### **Análise Exemplo 2.1**

* **Tempo por Iteração:** Os métodos possuem tempos por iteração parecidos.
* **Tempo Total:** O método do **Ponto Fixo Adaptado** e da **Secante Adaptada** possuem tempos totais parecidos.
* **Ineficiência:** O **Newton Polinomial** apresentou o pior tempo total, sendo mais de duas vezes mais lento do que os outros métodos.
* **Número de Operações:** O **Ponto Fixo Adaptado** é o método de menor complexidade intrínseca, realizando apenas **1 operação** por iteração.
* **Avaliações de Função:** O método do **Ponto Fixo Adaptado** e da **Secante Adaptada** possuem os maiores números de avaliações totais (**1**).
* **Decisões Lógicas:** O **Newton Polinomial** é o método que mais sobrecarrega o fluxo lógico, com **35 decisões** totais, ao contrário do Ponto Fixo, que precisou de apenas **11 decisões** no total.

---

### **Exemplo 2.2**

In [20]:
coefficients_example_2 = [1, 0, -3, 3][::-1]
fixed_point_iteration_function_example_2 = lambda x: math.cbrt(3*x - 3)
stopping_crit = 10**-6

newton_numerical_results, newton_effort_results, newton_time_results = newton_horner(coefficients_example_2, -2, stopping_crit, 4, max_iterations=10)

secant_numerical_results, secant_effort_results, secant_time_results = secant_horner(coefficients_example_2, -2, -2.1, stopping_crit, 4, max_iterations=10)

fixed_point_numerical_results, fixed_point_effort_results, fixed_point_time_results = fixed_point_horner(coefficients_example_2, fixed_point_iteration_function_example_2, -2, stopping_crit, 4, max_iterations=10)

numerical_data = {
    "newton": newton_numerical_results, 
    "secant": secant_numerical_results, 
    "fixed_point": fixed_point_numerical_results
}

effort_data = {
    "newton": newton_effort_results, 
    "secant": secant_effort_results, 
    "fixed_point": fixed_point_effort_results
}

time_data = {
    "newton": newton_time_results, 
    "secant": secant_time_results, 
    "fixed_point": fixed_point_time_results
}

print_task_b_tables(numerical_data, effort_data, time_data)

## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Dados Iniciais,X0 = -2,"X0=-2, X1=-2.1",X0 = -2
x̄,-2.103803,-2.103803,-2.103803
f(x̄),-6.6975 x 10^-09,6.1280 x 10^-06,1.6610 x 10^-06
Erro em x,3.2575 x 10^-05,5.9614 x 10^-07,5.5366 x 10^-07
Número de Iterações,4,3,9


## Tabela 2 – Análise de Esforço Computacional

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Operações por Iteração,11,7,1
Complexidade de Operação,O(3),O(3),O(1)
Decisões Lógicas Totais,7,6,9
Avaliações de Função por Iteração,0,1,1
Número de Iterações,4,3,9


## Tabela 3 – Análise de Tempo de Execução

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Tempo por Iteração (ms),0.002025,0.001267,0.000667
Tempo Total (ms),0.008100,0.003800,0.006000


#### **Análise Exemplo 2.2**

* **Tempo por Iteração:** O método do **Ponto Fixo Adaptado** e da Secante Adaptada possuem tempos por iteração parecidos.
* **Tempo Total:** O método da **Secante Adaptada** foi o mais rápido em tempo total, precisando de apenas **3 iterações** para convergir.
* **Ineficiência:** O **Newton Polinomial** e o **Ponto Fixo Adaptada** possuem tempos totais parecidos, sendo mais de duas vezes mais lentos que o método da Secante Adaptada.
* **Número de Operações:** O **Ponto Fixo Adaptado** é o método de menor complexidade intrínseca, realizando apenas **1 operação** por iteração, comparado a **11 operações** do Newton.
* **Avaliações de Função:** O método do **Ponto Fixo Adaptado** e da **Secante Adaptada** possuem os maiores números de avaliações totais por iteração (**1**).
* **Decisões Lógicas:** O **Newton Polinomial** é o método que mais sobrecarrega o fluxo lógico, com **7 decisões** totais, ao contrário da Secante Adaptada, que precisou de apenas **6 decisões** no total.